In [1]:
# Imports

import glob
import json
import os
import platform
import re

from collections import Counter, defaultdict

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import tensorflow as tf

from music21 import (
    converter,
    note,
    pitch,
)


# GPU Memory Growth
gpus = tf.config.list_physical_devices("GPU")

if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

In [2]:
# Sequenzlänge, Fenstergröße, realiver Postionsparameter

seq_len = 1026
step_size = 513
max_dist = 256   

batch_size = 16


In [3]:
# Special Vorzeichen werden gemappt, weil diese nicht in Music21 vorkommen

def load_score_safe(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        xml_text = f.read()

    # Flats
    xml_text = xml_text.replace("slash-flat", "quarter-flat")
    xml_text = xml_text.replace("three-quarters-flat", "quarter-flat")

    # Sharps 
    xml_text = xml_text.replace("slash-quarter-sharp", "quarter-sharp")
    xml_text = xml_text.replace("slash-sharp", "quarter-sharp")  
    xml_text = xml_text.replace("three-quarters-sharp", "quarter-sharp")  

    return converter.parseData(xml_text)

In [4]:
# ==========================================================
# Datenpfade setzen und XML-Dateien laden
# ==========================================================

system = platform.system()

DATA_PATHS = {
    "Windows": {
        "train": r"K:\Datensätze\DS_1\split_80_20\Train_2",
        "val":   r"K:\Datensätze\DS_1\split_80_20\validation",
        "test":  r"K:\Datensätze\DS_1\split_80_20\Test",
    },
    "Linux": {
        "train": "/home/users/d/den_oez/datasets/DS_1/split_80_20/Train_2",
        "val":   "/home/users/d/den_oez/datasets/DS_1/split_80_20/validation",
        "test":  "/home/users/d/den_oez/datasets/DS_1/split_80_20/Test",
    },
}

if system not in DATA_PATHS:
    raise RuntimeError(f"Unsupported system: {system}")

DATA_TRAIN = DATA_PATHS[system]["train"]
DATA_VAL   = DATA_PATHS[system]["val"]
DATA_TEST  = DATA_PATHS[system]["test"]


def find_xml_files(path, name):
    files = sorted(glob.glob(os.path.join(path, "**", "*.xml"), recursive=True))
    if not files:
        raise RuntimeError(f"Keine {name}-XML-Dateien gefunden: {path}")
    return files


xml_files_train = find_xml_files(DATA_TRAIN, "Train")
xml_files_validation = find_xml_files(DATA_VAL, "Validation")
xml_files_test = find_xml_files(DATA_TEST, "Test")

score_train = load_score_safe(xml_files_train[0])
score_val = load_score_safe(xml_files_validation[0])
score_test = load_score_safe(xml_files_test[0])

print("System:", system)
print("Train:", DATA_TRAIN, len(xml_files_train))
print("Val:", DATA_VAL, len(xml_files_validation))
print("Test:", DATA_TEST, len(xml_files_test))
print("Erste Dateien:")
print(xml_files_train[0])
print(xml_files_validation[0])
print(xml_files_test[0])

System: Windows
Train-Ordner: K:\Datensätze\DS_1\split_80_20\Train_2
Validation-Ordner: K:\Datensätze\DS_1\split_80_20\validation
Test-Ordner: K:\Datensätze\DS_1\split_80_20\Test

Train XML-Dateien: 1470
Validation XML-Dateien: 404
Test XML-Dateien: 172

Erste Train-Datei: K:\Datensätze\DS_1\split_80_20\Train_2\000000__hicaz--sarki--aksak--ben_gamli_hazan--melahat_pars.xml
Erste Validation-Datei: K:\Datensätze\DS_1\split_80_20\validation\000000__acem--ilahi--nimevsat--calabim_bir--haci_bayram_veli.xml
Erste Test-Datei: K:\Datensätze\DS_1\split_80_20\Test\000019__suzinak--pesrev--cenber----tatyos_efendi.xml


In [5]:
# Hier Wird der Makam-Namen aus XML-Dateinamen extrahiert
# und als Token-Dictionary gespeichert

def extract_makam_from_filename(filepath):

    filename = os.path.basename(filepath)
    name = os.path.splitext(filename)[0].lower()

    # Beispiel:
    # 000012__rast--eser

    if "__" not in name:
        return None

    after_id = name.split("__", 1)[1]

    if "--" in after_id:
        makam = after_id.split("--", 1)[0]
    else:
        makam = after_id

    return makam.strip()


def build_makam_keywords_from_files(xml_files_train):

    unique_makams = sorted({
        extract_makam_from_filename(filepath)
        for filepath in xml_files_train
        if extract_makam_from_filename(filepath) is not None
    })

    return {
        makam: f"<{makam.upper()}>"
        for makam in unique_makams
    }


MAKAM_KEYWORDS = build_makam_keywords_from_files(xml_files_train)

print(MAKAM_KEYWORDS)

MAKAM_TOKENS = list(MAKAM_KEYWORDS.values())
print(MAKAM_TOKENS)

{'acem': '<ACEM>', 'acemasiran': '<ACEMASIRAN>', 'acemkurdi': '<ACEMKURDI>', 'bestenigar': '<BESTENIGAR>', 'beyati': '<BEYATI>', 'beyati_araban': '<BEYATI_ARABAN>', 'buselik': '<BUSELIK>', 'dugah': '<DUGAH>', 'evcara': '<EVCARA>', 'evic': '<EVIC>', 'ferahfeza': '<FERAHFEZA>', 'ferahnak': '<FERAHNAK>', 'gerdaniye': '<GERDANIYE>', 'gulizar': '<GULIZAR>', 'hicaz': '<HICAZ>', 'hicaz_humayun': '<HICAZ_HUMAYUN>', 'hicaz_uzzal': '<HICAZ_UZZAL>', 'hicaz_zirgule': '<HICAZ_ZIRGULE>', 'hicazkar': '<HICAZKAR>', 'hisar': '<HISAR>', 'hisarbuselik': '<HISARBUSELIK>', 'huseyni': '<HUSEYNI>', 'huseyniasiran': '<HUSEYNIASIRAN>', 'huzzam': '<HUZZAM>', 'irak': '<IRAK>', 'isfahan': '<ISFAHAN>', 'karcigar': '<KARCIGAR>', 'kurdi': '<KURDI>', 'kurdilihicazkar': '<KURDILIHICAZKAR>', 'mahur': '<MAHUR>', 'muhayyer': '<MUHAYYER>', 'muhayyerkurdi': '<MUHAYYERKURDI>', 'mustear': '<MUSTEAR>', 'neva': '<NEVA>', 'neveser': '<NEVESER>', 'nihavent': '<NIHAVENT>', 'nikriz': '<NIKRIZ>', 'nisaburek': '<NISABUREK>', 'rast':

In [6]:
# Noten + Pausen werden aus music21-Score extrahiert
# Grace und Ornamente werden übersprungen
# Ergebnisse werden in "Events" gespeichert 

def extract_melody_notes_and_rests(score):
    part = score.parts[0] #erste Stimme
    flattened_part = part.flatten() #Entfernung der Hierarchie

    events = []
    for element in flattened_part:

        # ===== NOTE =====
        if isinstance(element, note.Note): #prüft ob Note vorhanden
            if element.duration.isGrace: #prüft ob Verzierungen/Ornament vorhanden
                continue
            events.append(element) #events hinzufügen

        # ===== REST =====
        elif isinstance(element, note.Rest):
            events.append(element)

    return events


In [7]:
# Kontrollausgabe: Noten + Accidentals + Duration + Oktave
for n in score_train.recurse().notes:
    acc = n.pitch.accidental
    alter = acc.alter if acc else 0

    print(
        "pitch:", n.pitch.nameWithOctave,
        "| step:", n.pitch.step,
        "| oct:", n.pitch.octave,
        "| alter:", alter,
        "| dur:", n.duration.quarterLength
    )

pitch: A4 | step: A | oct: 4 | alter: 0 | dur: 1.0
pitch: C#5 | step: C | oct: 5 | alter: 1.0 | dur: 1.0
pitch: B`4 | step: B | oct: 4 | alter: -0.5 | dur: 0.5
pitch: C#5 | step: C | oct: 5 | alter: 1.0 | dur: 0.5
pitch: A4 | step: A | oct: 4 | alter: 0 | dur: 1.5
pitch: D5 | step: D | oct: 5 | alter: 0 | dur: 1.0
pitch: C#5 | step: C | oct: 5 | alter: 1.0 | dur: 0.5
pitch: D5 | step: D | oct: 5 | alter: 0 | dur: 0.5
pitch: E5 | step: E | oct: 5 | alter: 0 | dur: 2.0
pitch: A5 | step: A | oct: 5 | alter: 0 | dur: 0.5
pitch: G5 | step: G | oct: 5 | alter: 0 | dur: 0.5
pitch: F5 | step: F | oct: 5 | alter: 0 | dur: 0.5
pitch: E5 | step: E | oct: 5 | alter: 0 | dur: 0.5
pitch: D5 | step: D | oct: 5 | alter: 0 | dur: 0.5
pitch: C#5 | step: C | oct: 5 | alter: 1.0 | dur: 0.5
pitch: B`4 | step: B | oct: 4 | alter: -0.5 | dur: 0.5
pitch: A4 | step: A | oct: 4 | alter: 0 | dur: 0.5
pitch: B`4 | step: B | oct: 4 | alter: -0.5 | dur: 0.5
pitch: C#5 | step: C | oct: 5 | alter: 1.0 | dur: 0.5
pitc

In [8]:
# Special-Vorzeichen werden hier gemappt

def is_special_accidental(acc):
    return acc in {
        "slash-flat",
        "three-quarters-flat",
        "slash-quarter-sharp",
        "slash-sharp",
        "three-quarters-sharp",
    }


def extract_original_accidentals(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        xml_text = f.read()

    return re.findall(r"<accidental>(.*?)</accidental>", xml_text)

In [9]:
# Analyseblock: Ersten Events der Trainingsdateien anzeigen 

events_train = extract_melody_notes_and_rests(score_train)
original_accidentals = extract_original_accidentals(current_xml_train)

print("Anzahl gefundener Events (Train):", len(events_train))
print("\nErste 25 Train-Events:")

acc_counter = 0  # zählt nur Noten mit accidental im XML

for event in events_train[:10]:

    if isinstance(event, note.Note):
        step = event.pitch.step
        octave = event.pitch.octave
        duration = event.quarterLength
        accidental = event.pitch.accidental

        if accidental:
            # passende XML-Accidental holen
            if acc_counter < len(original_accidentals):
                current_acc = original_accidentals[acc_counter]
            else:
                current_acc = None

            acc_counter += 1

            # prüfen ob special
            if is_special_accidental(current_acc):
                print(f"NOTE  {step}{octave}  alter=SPECIAL  dur={duration}")
            else:
                alter = accidental.alter
                print(f"NOTE  {step}{octave}  alter={alter:>5}  dur={duration}")

        else:
            print(f"NOTE  {step}{octave}  alter=0      dur={duration}")

    elif isinstance(event, note.Rest):
        duration = event.quarterLength
        print(f"REST            dur={duration}")

Anzahl gefundener Events (Train): 351

Erste 25 Train-Events:
NOTE  A4  alter=0      dur=1.0
NOTE  C5  alter=  1.0  dur=1.0
NOTE  B4  alter=SPECIAL  dur=0.5
NOTE  C5  alter=  1.0  dur=0.5
NOTE  A4  alter=0      dur=1.5
NOTE  D5  alter=0      dur=1.0
NOTE  C5  alter=  1.0  dur=0.5
NOTE  D5  alter=0      dur=0.5
NOTE  E5  alter=0      dur=2.0
NOTE  A5  alter=0      dur=0.5


events_val = extract_melody_notes_and_rests_train(score_val)

print("Anzahl gefundener Events (Train):", len(events_val))
print("Musikstück:", current_xml_val)

print("\nErste 25 Train-Events:")
for event in events_val[:25]:
    if isinstance(event, note.Note): #wenn das Event eine Note ist
        step = event.pitch.step #Tonstufe auslesen
        alter = event.pitch.accidental.alter if event.pitch.accidental else 0
        
        dur = event.quarterLength
        print(f"NOTE  {step:2s}  alter={alter:>5}  dur={dur}")

    elif isinstance(events, note.Rest):
        dur = event.quarterLength
        print(f"REST            dur={dur}")


In [10]:
# Vorzeichen Mapping

def alter_to_koma(alter, original_accidental=None):
    special_mapping = {
        "quarter-flat": -1,
        "slash-flat": -4,
        "slash-quarter-flat": -5,
        "three-quarters-flat": -8,

        "quarter-sharp": 1,
        "slash-sharp": 4,
        "slash-quarter-sharp": 5,
        "three-quarters-sharp": 8,
    }

    

    if original_accidental in special_mapping:
        return special_mapping[original_accidental]

    mapping = {
        -1.0: -4,
        -0.5: -1,
         0.0: 0,
         0.5: 1,
         1.0: 4,
    }

    if alter not in mapping:
        raise ValueError(f"Unbekannter alter-Wert: {alter}")

    return mapping[alter]

# Mittelpunkt/Referenzpunkt wird auf Rast = 0 gesetzt

STEP_TO_PERDE = {
    "G": "RAST",
    "A": "DUGAH",
    "B": "BUSELIK",
    "C": "CARGAH",
    "D": "NEVA",
    "E": "HUSEYNI",
    "F": "ACEM",
}

PERDE_BASE_KOMA = {
    "RAST": 0,
    "DUGAH": 9,
    "BUSELIK": 18,
    "CARGAH": 22,
    "NEVA": 31,
    "HUSEYNI": 40,
    "ACEM": 44,
}

In [11]:
# Absolute-Koma-Wert wird hier extrahiert
# music21-Note in Koma-Tonklasse umwandeln

def note_to_pitchclass_koma(event, xml_accidental=None):
    step = event.pitch.step
    alter = event.pitch.accidental.alter if event.pitch.accidental else 0.0
    perde = STEP_TO_PERDE.get(step)

    if perde is None:
        return None

    koma = PERDE_BASE_KOMA[perde] + alter_to_koma(alter, xml_accidental)

    return koma


def note_to_absolute_koma(event, xml_accidental=None):
    pitchclass_koma = note_to_pitchclass_koma(event, xml_accidental)

    if pitchclass_koma is None:
        return None

    octave = event.pitch.octave

    if octave is None:
        return None

    if event.pitch.step in ["G", "A", "B"]:
        makam_octave = octave
    else:
        makam_octave = octave - 1

    return pitchclass_koma + makam_octave * 53

In [12]:
# Debug-Ausgabe Zelle

events_train = extract_melody_notes_and_rests(score_train)
original_accidentals = extract_original_accidentals(current_xml_train)

print("=== DEBUG: Vorzeichen & Koma ===\n")

acc_counter = 0

for i, event in enumerate(events_train[:20], start=1):

    if isinstance(event, note.Note):

        step = event.pitch.step
        octave = event.pitch.octave
        duration = event.quarterLength

        accidental = event.pitch.accidental
        acc_name = accidental.name if accidental else "natural"
        alter = accidental.alter if accidental else 0.0

        # Original XML-Accidental holen
        if accidental:
            if acc_counter < len(original_accidentals):
                original_acc = original_accidentals[acc_counter]
            else:
                original_acc = None
            acc_counter += 1
        else:
            original_acc = None

        # Koma berechnen
        try:
            pitchclass_koma = note_to_pitchclass_koma(event, original_acc)
            absolute_koma = note_to_absolute_koma(event, original_acc)
        except Exception as e:
            pitchclass_koma = f"ERROR ({e})"
            absolute_koma = f"ERROR ({e})"

        print(
            f"{i:02d} | "
            f"{step:2s} | "
            f"oct={octave} | "
            f"xml={str(original_acc):20s} | "
            f"m21={acc_name:12s} | "
            f"alter={alter:5} | "
            f"pc_koma={pitchclass_koma} | "
            f"abs_koma={absolute_koma} | "
            f"dur={duration}"
        )

    elif isinstance(event, note.Rest):
        duration = event.quarterLength
        print(f"{i:02d} | REST | dur={duration}")

=== DEBUG: Vorzeichen & Koma ===

01 | A  | oct=4 | xml=None                 | m21=natural      | alter=  0.0 | pc_koma=9 | abs_koma=221 | dur=1.0
02 | C  | oct=5 | xml=sharp                | m21=sharp        | alter=  1.0 | pc_koma=26 | abs_koma=238 | dur=1.0
03 | B  | oct=4 | xml=slash-flat           | m21=half-flat    | alter= -0.5 | pc_koma=14 | abs_koma=226 | dur=0.5
04 | C  | oct=5 | xml=sharp                | m21=sharp        | alter=  1.0 | pc_koma=26 | abs_koma=238 | dur=0.5
05 | A  | oct=4 | xml=None                 | m21=natural      | alter=  0.0 | pc_koma=9 | abs_koma=221 | dur=1.5
06 | D  | oct=5 | xml=None                 | m21=natural      | alter=  0.0 | pc_koma=31 | abs_koma=243 | dur=1.0
07 | C  | oct=5 | xml=sharp                | m21=sharp        | alter=  1.0 | pc_koma=26 | abs_koma=238 | dur=0.5
08 | D  | oct=5 | xml=None                 | m21=natural      | alter=  0.0 | pc_koma=31 | abs_koma=243 | dur=0.5
09 | E  | oct=5 | xml=None                 | m21=natural

In [13]:
# Tonikabestimmung: letzte Note eines Stücks

def detect_tonic_duration_weighted_m21(events, original_accidentals=None):
    if not events:
        raise ValueError("Leere Eventliste")

    note_events = [event for event in events if isinstance(event, note.Note)]
    if not note_events:
        raise ValueError("Keine Noten vorhanden")

    note_komas = []
    acc_idx = 0

    for event in note_events:
        step = event.pitch.step
        octave = event.pitch.octave
        alter = event.pitch.accidental.alter if event.pitch.accidental else 0.0

        original_acc = None
        if event.pitch.accidental:
            if original_accidentals is not None and acc_idx < len(original_accidentals):
                original_acc = original_accidentals[acc_idx]
            acc_idx += 1

        koma = note_to_pitchclass_koma(event, original_acc)
        if koma is None:
            continue

        note_komas.append({
            "event": event,
            "step": step,
            "octave": octave,
            "alter": alter,
            "original_accidental": original_acc,
            "koma": koma,
        })

    if not note_komas:
        raise ValueError("Keine gültigen Koma-Werte gefunden")

    # Tonika = letzte gültige Note
    last_note = note_komas[-1]

    tonic_step = last_note["step"]
    tonic_octave = last_note["octave"]
    tonic_alter = last_note["alter"]
    tonic_koma = last_note["koma"]

    max_score = float(last_note["event"].quarterLength)

    return tonic_step, tonic_octave, tonic_alter, tonic_koma, max_score

In [14]:
# Debug/Ausgabe Zelle

tonic_step_train, tonic_octave_train, tonic_alter_train, tonic_koma_train, tonic_score_train = \
    detect_tonic_duration_weighted_m21(
        events_train,
        original_accidentals
    )

print(
    "Teststück Train-Tonika:",
    f"{tonic_step_train}{tonic_octave_train}",
    "| alter =", tonic_alter_train,
    "| pc_koma =", tonic_koma_train,
    "| score =", round(tonic_score_train, 2)
)

Teststück Train-Tonika: A4 | alter = 0.0 | pc_koma = 9 | score = 1.5


In [15]:
# Debug: letzte Note = Tonika

from music21 import note

acc_idx = 0
last_note = None

for event in events_train:

    if not isinstance(event, note.Note):
        continue

    original_acc = None

    if event.pitch.accidental:
        if acc_idx < len(original_accidentals):
            original_acc = original_accidentals[acc_idx]
        acc_idx += 1

    pc_koma = note_to_pitchclass_koma(event, original_acc)
    abs_koma = note_to_absolute_koma(event, original_acc)

    if pc_koma is None:
        continue

    last_note = {
        "step": event.pitch.step,
        "octave": event.pitch.octave,
        "alter": event.pitch.accidental.alter if event.pitch.accidental else 0.0,
        "pc_koma": pc_koma,
        "abs_koma": abs_koma,
        "duration": float(event.quarterLength),
    }

if last_note is None:
    raise ValueError("Keine gültige Note gefunden")

print("\n--- Tonika aus letzter Note ---")

print(
    f"{last_note['step']}{last_note['octave']}",
    "| alter =", last_note["alter"],
    "| pc_koma =", last_note["pc_koma"],
    "| abs_koma =", last_note["abs_koma"],
    "| duration =", round(last_note["duration"], 2)
)


--- Tonika aus letzter Note ---
A4 | alter = 0.0 | pc_koma = 9 | abs_koma = 221 | duration = 1.5


In [16]:
# Tonika werden über alle Musikstücke berechnet und gespeichert

tonics = {}

for xml_path in xml_files_train:

    makam_typ = extract_makam_from_filename(xml_path)

    if makam_typ is None:
        continue

    makam_token = MAKAM_KEYWORDS.get(makam_typ)

    if makam_token is None:
        print(
            "Unbekannter Makam:",
            makam_typ,
            "| Datei:",
            os.path.basename(xml_path)
        )
        continue

    try:
        score = load_score_safe(xml_path)

        events = extract_melody_notes_and_rests(score)

        if not events:
            continue

        original_accidentals = extract_original_accidentals(xml_path)

        tonic_step, tonic_octave, tonic_alter, tonic_pc_koma, tonic_score = \
            detect_tonic_duration_weighted_m21(
                events,
                original_accidentals
            )

        print(
            f"{os.path.basename(xml_path)} | "
            f"{makam_token} | "
            f"Tonika: {tonic_step}{tonic_octave} | "
            f"alter={tonic_alter} | "
            f"pc_koma={tonic_pc_koma} | "
            f"score={round(tonic_score, 2)}"
        )

        tonics[xml_path] = {
            "makam_token": makam_token,
            "makam_typ": makam_typ,
            "tonic_step": tonic_step,
            "tonic_octave": tonic_octave,
            "tonic_alter": tonic_alter,
            "tonic_pc_koma": tonic_pc_koma,
            "tonic_score": tonic_score,
        }

    except Exception as exc:

        print(
            "Fehler bei",
            os.path.basename(xml_path),
            ":",
            exc
        )

000000__hicaz--sarki--aksak--ben_gamli_hazan--melahat_pars.xml | <HICAZ> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.5
000001__rast--rumeliturkusu--devrihindi_ii--kircaliyle_arda--rumeli.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.0
000002__nihavent--sarki--aksak--gece_sahilden--serif_icli.xml | <NIHAVENT> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.5
000003__rast--sarki--curcuna--nihansin_dideden--haci_faik_bey.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=2.0
000004__buselik--sarki--agiraksak--bir_peri-ruyin--hamparsum.xml | <BUSELIK> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.5
000005__beyati--sarki--duyek--askn_fecri--sadettin_kaynak.xml | <BEYATI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=2.0
000006__muhayyer--sarki--yuruksemai--ben_sana_asik--dede_efendi.xml | <MUHAYYER> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=3.0
000007__huseyni--turku--devrituran--yagmur_yagar--.xml | <HUSEYNI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=3.5
000008

000071__acemasiran--sarki--semai--ey_cesm-i--nikogos_aga.xml | <ACEMASIRAN> | Tonika: F4 | alter=0.0 | pc_koma=44 | score=3.0
000072__huseyni--sarki--devrihindi--gormek_ister--tanburi_cemil_bey.xml | <HUSEYNI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
000073__yeni_cargah--turku--sofyan--bilmem_su--.xml | <YENI_CARGAH> | Tonika: C5 | alter=0.0 | pc_koma=22 | score=0.5
000074__neva--sazsemaisi--aksaksemai----kantemiroglu.xml | <NEVA> | Tonika: D5 | alter=0.0 | pc_koma=31 | score=1.0
000075__beyati--beste--hafif--bir_gonca_femin--dede_efendi.xml | <BEYATI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=2.0
000076__gulizar--turku--sofyan--dertli--erol_sayan.xml | <GULIZAR> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
000077__hicaz_humayun--pesrev--cenber----veli_dede.xml | <HICAZ_HUMAYUN> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
000078__hicaz_humayun--turku--turkaksagi--beni_seni--trabzon.xml | <HICAZ_HUMAYUN> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.5
000079__segah-

000145__mahur--sazsemaisi--aksaksemai--bahar_1--goksel_baktagir.xml | <MAHUR> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.0
000146__acemasiran--sarki--duyek--ruhumda_derin--kemal_gurses.xml | <ACEMASIRAN> | Tonika: F4 | alter=0.0 | pc_koma=44 | score=1.0
000147__isfahan--pesrev--devrikebir----tanburi_cemil_bey.xml | <ISFAHAN> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=2.0
000148__muhayyer--sarki--aksak--iltimas_etmeye--haci_arif_bey.xml | <MUHAYYER> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
000149__suzinak--seyir--semai--1--rauf_yekta.xml | <SUZINAK> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=2.0
000150__buselik--murabba--zencir--gonul_o_turra--yahya_nazim_celebi.xml | <BUSELIK> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=3.0
000151__suzinak_zirgule--sarki--devrihindi--dogdugum_gunden--muallim_ismail_hakki_bey.xml | <SUZINAK_ZIRGULE> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.0
000152__kurdilihicazkar--sarki--curcuna--bu_aksam--avni_anil.xml | <KURDILIHICAZKAR> | Ton

000218__hicaz--sarki--agiraksak--cevr-i_hicrin--bulbuli_salih_efendi.xml | <HICAZ> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=2.0
000219__rast--sarki--semai--yine_bir_gulnihal--dede_efendi.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=2.0
000220__rast--pesrev--sakil----benli_hasan_aga.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=3.0
000221__hicaz--sarki--curcuna--acaba_sen_misin--bimen_sen.xml | <HICAZ> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
000222__karcigar--mandra--devrituran--karadeniz_oyunhavasi--.xml | <KARCIGAR> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
000223__huzzam--sarki--agiraksak--neseyab-i_lutfun--musullu_hafiz_osman_efendi.xml | <HUZZAM> | Tonika: B4 | alter=-0.5 | pc_koma=17 | score=1.0
000224__mahur--sarki--agiraksak--var_iken--tanburi_cemil_bey.xml | <MAHUR> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.5
000225__kurdilihicazkar--sazsemaisi--aksaksemai----haydar_tatliyay.xml | <KURDILIHICAZKAR> | Tonika: G4 | alter=0.0 | pc_

000288__kurdilihicazkar--sarki--duyek--gecmesin_gunumuz--alaeddin_yavasca.xml | <KURDILIHICAZKAR> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=4.0
000289__hisarbuselik--turku--curcuna--araz_uste--.xml | <HISARBUSELIK> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.5
000290__nisaburek--sarki--turkaksagi--bir_soz_dedi--munir_nurettin_selcuk.xml | <NISABUREK> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
000292__hicazkar--kocekce--aksak--gece_gunduz--muallim_ismail_hakki_bey.xml | <HICAZKAR> | Tonika: G5 | alter=0.0 | pc_koma=0 | score=1.0
000293__muhayyerkurdi--sarki--nimsofyan--gonlumu_gonlune--omer_sami_gupgup.xml | <MUHAYYERKURDI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
000294__rast--sarki--duyek--saclarin_tarumar--sekip_ayhan_ozisik.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=3.5
000295__rast--etud--sofyan----ozer_ozel.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.0
000296__segah--rumeliturkusu--raksaksagi--sana_da--rumeli.xml | <SEGAH> | Tonik

...

001332__nihavent--sarki--duyek--seninle_bu--dursun_karaca.xml | <NIHAVENT> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=4.0
001333__muhayyerkurdi--sarki--sofyan--mevsimler_yas--yildirim_gurses.xml | <MUHAYYERKURDI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=4.0
001334__huzzam--sarki--semai--her_gece--kasim_inaltekin.xml | <HUZZAM> | Tonika: B4 | alter=-0.5 | pc_koma=17 | score=1.0
001335__gerdaniye--rumeliturkusu--dolap--kirimdan_gelirim--rumeli.xml | <GERDANIYE> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=3.0
001336__acemkurdi--sarki--agiraksak--sevdi_gonlum--nikogos_aga.xml | <ACEMKURDI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=2.0
001337__segah--sarki--murekkepsofyan--dustu_enginlere--refik_fersan.xml | <SEGAH> | Tonika: B4 | alter=-0.5 | pc_koma=17 | score=4.0
001338__acemkurdi--sarki--sofyan--yar_pesinde--zeki_duygulu.xml | <ACEMKURDI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=2.0
001339__nihavent--sarki--sofyan--korfezde_meltem--ismail_otenkaya.xml | <NIHAVENT> | Tonika

001405__sultaniyegah--sarki--curcuna--gozumde_hep--.xml | <SULTANIYEGAH> | Tonika: D4 | alter=0.0 | pc_koma=31 | score=1.5
001406__rast--sarki--sofyan--acilan_bir--dramali_hasan_hasguler.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.0
001407__huzzam--sarki--duyek--pisman_olur_da--irfan_ozbakir.xml | <HUZZAM> | Tonika: B4 | alter=-0.5 | pc_koma=17 | score=3.0
001408__yeni_cargah--seyir--sofyan--1--erol_bingol.xml | <YENI_CARGAH> | Tonika: C5 | alter=0.0 | pc_koma=22 | score=2.0
001409__hicaz_uzzal--yuruksemai--yuruksemai--terk_eyledi--zaharya.xml | <HICAZ_UZZAL> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=3.0
001410__mahur--sarki--duyek--olsa_da_hos--sadettin_kaynak.xml | <MAHUR> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.0
001411__saba--sarki--senginsemai--hasretle_anarken--yesari_asim_arsoy.xml | <SABA> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=2.0
001412__kurdilihicazkar--sarki--duyek--gittin_biraktin--emin_ongan.xml | <KURDILIHICAZKAR> | Tonika: G4 | alter=0.0 

001478__rast--sarki--aksak--aski_huznumle--mildan_niyazi_bey.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=2.0
001479__rast--turku--nimsofyan--ibrisim_ormuyorlar--.xml | <RAST> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=2.0
001480__hicaz--turku--devrihindi--cayelinden_oteye--rize.xml | <HICAZ> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.0
001481__segah--fantezi--aksak--gul_ne--omer_dilek.xml | <SEGAH> | Tonika: B4 | alter=-0.5 | pc_koma=17 | score=2.0
001482__tahir--turku--ciftesofyan--yayla_yollarinda--.xml | <TAHIR> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=0.25
001483__buselik--seyir--sofyan--1--sefik_gurmeric.xml | <BUSELIK> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=4.0
001484__nihavent--pesrev--hafif--mini_mini--huseyin_sadettin_arel.xml | <NIHAVENT> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=2.0
001486__huseyniasiran--seyir--duyek--1--erol_bingol.xml | <HUSEYNIASIRAN> | Tonika: E4 | alter=0.0 | pc_koma=40 | score=2.0
001487__saba--turku--curcuna--yesil_ya

001559__hicaz--sarki--evfer--mah_yuzune--dede_efendi.xml | <HICAZ> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.5
001561__huseyni--turku--sofyan--ciktim_belen--mugla.xml | <HUSEYNI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=0.5
001562__nikriz--zeybek--agiraksak----izmir.xml | <NIKRIZ> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=1.0
001563__nikriz--sarki--turkaksagi--bir_destan--selahattin_icli.xml | <NIKRIZ> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=0.25
001564__beyati--sarki--curcuna--kalbim_yine--selahattin_pinar.xml | <BEYATI> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=1.5
001565__suzidilara--pesrev--agirduyek----iii_selim.xml | <SUZIDILARA> | Tonika: G4 | alter=0.0 | pc_koma=0 | score=3.0
001568__huzzam--sarki--agiraksak--sabrimi_gamzelerin--bimen_sen.xml | <HUZZAM> | Tonika: B4 | alter=-0.5 | pc_koma=17 | score=1.5
001570__buselik--pesrev--muhammes----nikolaki.xml | <BUSELIK> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=3.0
001571__muhayyer--turku--raksaksagi_ii--su_gel

001640__segah--sazsemaisi--aksaksemai----hizir_aga.xml | <SEGAH> | Tonika: B4 | alter=-0.5 | pc_koma=17 | score=1.0
001641__buselik--aranagme--duyek--1--.xml | <BUSELIK> | Tonika: A4 | alter=0.0 | pc_koma=9 | score=2.0


In [17]:
# Token-Sequenzen werden in feste Fenstergrößen geschitten
# und mit <END> / <PAD> aufgefüllt

def slice_token_sequence(tokens, seq_len, step_size, pad_token="<PAD>"):
    if not tokens:
        return []

    slices = []

    for start in range(0, len(tokens), step_size):
        chunk = tokens[start:start + seq_len]

        if len(chunk) < seq_len:
            chunk = chunk + ["<END>"] * ("<END>" not in chunk)
            chunk = chunk + [pad_token] * (seq_len - len(chunk))

        slices.append(chunk)

        if start + seq_len >= len(tokens):
            break

    return slices



def slice_event_tokens(tokens, seq_len, step_events, pad_token="<PAD>"):
    event_len = 4
    inner_len = seq_len - 2
    events_per_slice = inner_len // event_len

    if events_per_slice < 1:
        raise ValueError("seq_len ist zu klein")

    tokens = tokens[:len(tokens) - (len(tokens) % event_len)]

    slices = []
    step_tokens = step_events * event_len
    slice_tokens = events_per_slice * event_len

    for start in range(0, len(tokens), step_tokens):
        chunk = tokens[start:start + slice_tokens]

        if not chunk:
            continue

        seq = ["<START>"] + chunk + ["<END>"]
        seq += [pad_token] * (seq_len - len(seq))
        slices.append(seq)

        if start + slice_tokens >= len(tokens):
            break

    return slices

In [18]:
# Berechnung von Koma-Wert für Tonika

def tonic_to_absolute_koma(tonic_step, tonic_pc_koma, tonic_octave):
    makam_octave = tonic_octave if tonic_step in ["G", "A", "B"] else tonic_octave - 1
    return tonic_pc_koma + makam_octave * 53


# Relative Tonhöhe wird berechnet
def events_to_koma_sequence(events, tonic_step, tonic_pc_koma, tonic_octave, original_accidentals=None):
    if tonic_pc_koma is None:
        return []

    tonic_abs_koma = tonic_to_absolute_koma(tonic_step, tonic_pc_koma, tonic_octave)

    seq = []
    acc_idx = 0

    for event in events:
        dur = float(event.quarterLength)

        if dur <= 0:
            continue

        if isinstance(event, note.Rest):
            seq.append({
                "TYPE": "REST",
                "REL_KOMA": None,
                "ABS_KOMA": None,
                "TONIC_ABS_KOMA": tonic_abs_koma,
                "DUR": dur
            })
            continue

        if not isinstance(event, note.Note):
            continue

        original_acc = None

        if event.pitch.accidental:
            if original_accidentals and acc_idx < len(original_accidentals):
                original_acc = original_accidentals[acc_idx]
            acc_idx += 1

        abs_koma = note_to_absolute_koma(event, original_acc)

        if abs_koma is None:
            continue

        seq.append({
            "TYPE": "NOTE",
            "REL_KOMA": abs_koma - tonic_abs_koma,
            "ABS_KOMA": abs_koma,
            "TONIC_ABS_KOMA": tonic_abs_koma,
            "DUR": dur
        })

    return seq

In [19]:
# Debug-Ausgabe

original_accidentals = extract_original_accidentals(current_xml_train)

info = tonics[current_xml_train]

seq_train = events_to_koma_sequence(
    events_train,
    info["tonic_step"],
    info["tonic_pc_koma"],
    info["tonic_octave"],
    original_accidentals
)

print("Train-Stück:", os.path.basename(current_xml_train))
print("Tonika:", f"{info['tonic_step']}{info['tonic_octave']}")
print("Tonika-pc-Koma:", info["tonic_pc_koma"])

for i, event in enumerate(seq_train[:15], start=1):
    print(
        f"{i:03d} | {event['TYPE']:4s} | "
        f"REL={str(event['REL_KOMA']):>6} | "
        f"ABS={str(event['ABS_KOMA']):>6} | "
        f"DUR={event['DUR']}"
    )

Train-Stück: 000000__hicaz--sarki--aksak--ben_gamli_hazan--melahat_pars.xml
Tonika: A4
Tonika-pc-Koma: 9
001 | NOTE | REL=     0 | ABS=   221 | DUR=1.0
002 | NOTE | REL=    17 | ABS=   238 | DUR=1.0
003 | NOTE | REL=     5 | ABS=   226 | DUR=0.5
004 | NOTE | REL=    17 | ABS=   238 | DUR=0.5
005 | NOTE | REL=     0 | ABS=   221 | DUR=1.5
006 | NOTE | REL=    22 | ABS=   243 | DUR=1.0
007 | NOTE | REL=    17 | ABS=   238 | DUR=0.5
008 | NOTE | REL=    22 | ABS=   243 | DUR=0.5
009 | NOTE | REL=    31 | ABS=   252 | DUR=2.0
010 | NOTE | REL=    53 | ABS=   274 | DUR=0.5
011 | NOTE | REL=    44 | ABS=   265 | DUR=0.5
012 | NOTE | REL=    35 | ABS=   256 | DUR=0.5
013 | NOTE | REL=    31 | ABS=   252 | DUR=0.5
014 | NOTE | REL=    22 | ABS=   243 | DUR=0.5
015 | NOTE | REL=    17 | ABS=   238 | DUR=0.5


In [20]:
# Makam-Vokabular wird als Json gespeichert

def build_makam_id_mapping_train(train_tonics_dict):
    return {
        makam: idx
        for idx, makam in enumerate(
            sorted({info["makam_token"] for info in train_tonics_dict.values()})
        )
    }


def get_makam_id(makam_token):
    if makam_token not in makam_to_id:
        raise ValueError(f"Unbekannter Makam-Token: {makam_token}")
    return makam_to_id[makam_token]


makam_to_id = build_makam_id_mapping_train(tonics)

print("Makam-Token -> ID:", makam_to_id)

with open("makam_to_id.json", "w", encoding="utf-8") as f:
    json.dump(makam_to_id, f, ensure_ascii=False, indent=4)

Makam-Token -> ID: {'<ACEM>': 0, '<ACEMASIRAN>': 1, '<ACEMKURDI>': 2, '<BESTENIGAR>': 3, '<BEYATI>': 4, '<BEYATI_ARABAN>': 5, '<BUSELIK>': 6, '<DUGAH>': 7, '<EVCARA>': 8, '<EVIC>': 9, '<FERAHFEZA>': 10, '<FERAHNAK>': 11, '<GERDANIYE>': 12, '<GULIZAR>': 13, '<HICAZ>': 14, '<HICAZKAR>': 15, '<HICAZ_HUMAYUN>': 16, '<HICAZ_UZZAL>': 17, '<HICAZ_ZIRGULE>': 18, '<HISAR>': 19, '<HISARBUSELIK>': 20, '<HUSEYNI>': 21, '<HUSEYNIASIRAN>': 22, '<HUZZAM>': 23, '<IRAK>': 24, '<ISFAHAN>': 25, '<KARCIGAR>': 26, '<KURDI>': 27, '<KURDILIHICAZKAR>': 28, '<MAHUR>': 29, '<MUHAYYER>': 30, '<MUHAYYERKURDI>': 31, '<MUSTEAR>': 32, '<NEVA>': 33, '<NEVESER>': 34, '<NIHAVENT>': 35, '<NIKRIZ>': 36, '<NISABUREK>': 37, '<RAST>': 38, '<REHAVI>': 39, '<SABA>': 40, '<SEDARABAN>': 41, '<SEGAH>': 42, '<SEHNAZ>': 43, '<SEHNAZBUSELIK>': 44, '<SEVKEFZA>': 45, '<SULTANIYEGAH>': 46, '<SUZIDIL>': 47, '<SUZIDILARA>': 48, '<SUZINAK>': 49, '<SUZINAK_ZIRGULE>': 50, '<TAHIR>': 51, '<USSAK>': 52, '<YEGAH>': 53, '<YENI_CARGAH>': 54, '<

In [21]:
# Notendauern werden quantisiert
# seltene Dauern werden zusammengeführt

DUR_QUANT_MAP = {
    2.5: 3,
    4.5: 4,
    5.0: 4,
    6.0: 4,
}


def quantize_duration(d):
    if d is None:
        return None

    d = round(float(d), 6)

    if d >= 7:
        return 5

    return DUR_QUANT_MAP.get(d, d)


def q(x, nd=1):
    if x is None:
        return "NONE"

    x = round(float(x), nd)
    return int(x) if x.is_integer() else x

In [22]:

# Statistik der Notendauern

def collect_duration_stats(xml_files):
    total = Counter()
    notes = Counter()
    rests = Counter()

    for xml_path in xml_files:
        events = extract_melody_notes_and_rests(
            load_score_safe(xml_path)
        )

        for event in events:
            dur = round(float(event.quarterLength), 6)

            total[dur] += 1

            if isinstance(event, note.Note):
                notes[dur] += 1
            elif isinstance(event, note.Rest):
                rests[dur] += 1

    return total, notes, rests

In [23]:
# Debug-Ausgabe

dur_counter, note_counter, rest_counter = collect_duration_stats(xml_files_train)

for dur, count in dur_counter.most_common():
    print(
        f"DUR={dur:<8} | "
        f"total={count:<6} | "
        f"notes={note_counter[dur]:<6} | "
        f"rests={rest_counter[dur]:<6}"
    )

print("\nUnique durations:", len(dur_counter))

DUR=0.5      | total=240826 | notes=226614 | rests=14212 
DUR=0.25     | total=194217 | notes=193205 | rests=1012  
DUR=1.0      | total=91564  | notes=85045  | rests=6519  
DUR=0.75     | total=16863  | notes=16859  | rests=4     
DUR=1.5      | total=15220  | notes=15072  | rests=148   
DUR=2.0      | total=13229  | notes=12822  | rests=407   
DUR=0.125    | total=10687  | notes=10668  | rests=19    
DUR=0.166667 | total=6209   | notes=6209   | rests=0     
DUR=3.0      | total=3973   | notes=3854   | rests=119   
DUR=0.333333 | total=3536   | notes=3536   | rests=0     
DUR=0.375    | total=3535   | notes=3535   | rests=0     
DUR=4.0      | total=1319   | notes=952    | rests=367   
DUR=3.5      | total=393    | notes=355    | rests=38    
DUR=4.5      | total=179    | notes=0      | rests=179   
DUR=5.0      | total=158    | notes=0      | rests=158   
DUR=6.0      | total=105    | notes=25     | rests=80    
DUR=0.0625   | total=71     | notes=71     | rests=0     
DUR=1.75     |

In [24]:
from collections import Counter
from music21 import note

quant_dur_counter = Counter()

for xml_path in xml_files_train:

    score = load_score_safe(xml_path)

    events = extract_melody_notes_and_rests(score)

    for event in events:

        dur = float(event.quarterLength)

        qdur = quantize_duration(dur)

        quant_dur_counter[qdur] += 1


print("Quantisierte Duration-Werte")
print("============================")

for dur in sorted(quant_dur_counter.keys()):
    print(
        f"DUR={str(dur):8s} | "
        f"total={quant_dur_counter[dur]}"
    )

Quantisierte Duration-Werte
DUR=0.0      | total=6
DUR=0.0625   | total=71
DUR=0.083333 | total=26
DUR=0.125    | total=10687
DUR=0.166667 | total=6209
DUR=0.1875   | total=33
DUR=0.25     | total=194217
DUR=0.333333 | total=3536
DUR=0.375    | total=3535
DUR=0.5      | total=240826
DUR=0.666667 | total=26
DUR=0.75     | total=16863
DUR=0.875    | total=50
DUR=1.0      | total=91564
DUR=1.5      | total=15220
DUR=1.75     | total=66
DUR=2.0      | total=13229
DUR=2.25     | total=2
DUR=3.0      | total=4003
DUR=3.5      | total=393
DUR=4        | total=1761
DUR=5        | total=180
DUR=6.5      | total=3


In [26]:
# Tokenization

def sequence_to_tokens(seq):
    tokens = ["<START>"]

    for evt in seq:
        dur = q(quantize_duration(evt["DUR"]))

        if evt["TYPE"] == "REST":
            tokens += ["<REST>", f"DUR:{dur}", "<SEP>"]
        else:
            tokens += [f"REL_KOMA:{q(evt['REL_KOMA'])}", f"DUR:{dur}", "<SEP>"]

    return tokens + ["<END>"]

In [27]:
# Aus MusicXML werden Token-Sequenzen

def build_dataset_sequences(xml_files, seq_len=seq_len, step_events=32):
    dataset_sequences, dataset_makam_ids, sequence_meta = [], [], []

    for xml in xml_files:
        original_makam = extract_makam_from_filename(xml)
        makam_token = MAKAM_KEYWORDS.get(original_makam)

        if makam_token is None:
            continue

        try:
            events = extract_melody_notes_and_rests(load_score_safe(xml))
            if not events:
                continue

            original_accidentals = extract_original_accidentals(xml)

            tonic_step, tonic_octave, tonic_alter, tonic_pc_koma, tonic_score = \
                detect_tonic_duration_weighted_m21(events, original_accidentals)

            koma_seq = events_to_koma_sequence(
                events,
                tonic_step,
                tonic_pc_koma,
                tonic_octave,
                original_accidentals
            )

            music_tokens = sequence_to_tokens(koma_seq)[1:-1]

            raw_slices = slice_event_tokens(
                music_tokens,
                seq_len=seq_len,
                step_events=step_events
            )

            makam_id = get_makam_id(makam_token)

            for final_slice in raw_slices:
                dataset_sequences.append(final_slice)
                dataset_makam_ids.append(makam_id)

                sequence_meta.append({
                    "xml": xml,
                    "makam_token": makam_token,
                    "makam_id": makam_id,
                    "makam_orig": original_makam,
                    "koma_representation": "relative",
                    "tonic_step": tonic_step,
                    "tonic_octave": tonic_octave,
                    "tonic_alter": tonic_alter,
                    "tonic_pc_koma": tonic_pc_koma,
                    "tonic_score": tonic_score,

                })

        except Exception as exc:
            print("Fehler bei", os.path.basename(xml), ":", exc)

    return dataset_sequences, dataset_makam_ids, sequence_meta

In [28]:
# Erzeugung der Datensätze

dataset_sequences_train, dataset_makam_ids_train, sequence_meta_train = build_dataset_sequences(
    xml_files_train,
    seq_len,
    step_size
)

dataset_sequences_val, dataset_makam_ids_val, sequence_meta_val = build_dataset_sequences(
    xml_files_validation,
    seq_len,
    step_size
)

dataset_sequences_test, dataset_makam_ids_test, sequence_meta_test = build_dataset_sequences(
    xml_files_test,
    seq_len,
    step_size
)

In [45]:
print("Train sequences:", len(dataset_sequences_train))
print("Val sequences:", len(dataset_sequences_val))
print("Test sequences:", len(dataset_sequences_test))

print("Train makam ids:", len(dataset_makam_ids_train))
print("Val makam ids:", len(dataset_makam_ids_val))
print("Test makam ids:", len(dataset_makam_ids_test))

Train sequences: 1602
Val sequences: 185
Test sequences: 448
Train makam ids: 1602
Val makam ids: 185
Test makam ids: 448


In [29]:
# Kontrollausgabe


id_to_makam = {v: k for k, v in makam_to_id.items()}


def print_dataset_example(name, sequences, makam_ids, metas, i=0):
    if not sequences:
        print(name, ": keine Sequenzen")
        return

    meta = metas[i]

    print("\n" + "=" * 60)
    print(name, "Index:", i)
    print("Makam:", id_to_makam[makam_ids[i]])
    print("Datei:", meta["xml"])
    print("Tonika:", f"{meta['tonic_step']}{meta['tonic_octave']}")
    print("pc_koma:", meta["tonic_pc_koma"])
    print("Koma:", meta["koma_representation"])
    print("Tokens:")
    print(" ".join(sequences[i]))


print_dataset_example(
    "TRAIN",
    dataset_sequences_train,
    dataset_makam_ids_train,
    sequence_meta_train
)

print_dataset_example(
    "VALIDATION",
    dataset_sequences_val,
    dataset_makam_ids_val,
    sequence_meta_val
)

print_dataset_example(
    "TEST",
    dataset_sequences_test,
    dataset_makam_ids_test,
    sequence_meta_test
)


TRAIN Index: 0
Makam: <HICAZ>
Datei: K:\Datensätze\DS_1\split_80_20\Train_2\000000__hicaz--sarki--aksak--ben_gamli_hazan--melahat_pars.xml
Tonika: A4
pc_koma: 9
Koma: relative
Tokens:
<START> REL_KOMA:0 DUR:1 <SEP> REL_KOMA:17 DUR:1 <SEP> REL_KOMA:5 DUR:0.5 <SEP> REL_KOMA:17 DUR:0.5 <SEP> REL_KOMA:0 DUR:1.5 <SEP> REL_KOMA:22 DUR:1 <SEP> REL_KOMA:17 DUR:0.5 <SEP> REL_KOMA:22 DUR:0.5 <SEP> REL_KOMA:31 DUR:2 <SEP> REL_KOMA:53 DUR:0.5 <SEP> REL_KOMA:44 DUR:0.5 <SEP> REL_KOMA:35 DUR:0.5 <SEP> REL_KOMA:31 DUR:0.5 <SEP> REL_KOMA:22 DUR:0.5 <SEP> REL_KOMA:17 DUR:0.5 <SEP> REL_KOMA:5 DUR:0.5 <SEP> REL_KOMA:0 DUR:0.5 <SEP> REL_KOMA:5 DUR:0.5 <SEP> REL_KOMA:17 DUR:0.5 <SEP> REL_KOMA:0 DUR:1.5 <SEP> REL_KOMA:5 DUR:0.2 <SEP> REL_KOMA:17 DUR:0.2 <SEP> REL_KOMA:22 DUR:0.2 <SEP> REL_KOMA:22 DUR:0.2 <SEP> REL_KOMA:17 DUR:0.2 <SEP> REL_KOMA:5 DUR:0.2 <SEP> REL_KOMA:0 DUR:1 <SEP> <REST> DUR:0.5 <SEP> REL_KOMA:44 DUR:0.5 <SEP> REL_KOMA:35 DUR:0.2 <SEP> REL_KOMA:31 DUR:0.2 <SEP> REL_KOMA:22 DUR:0.5 <SEP> 

In [30]:
# Debug-Ausgabe

tok_counter_train = Counter()

for seq in dataset_sequences_train:
    tok_counter_train.update(seq)

print("TRAIN")
print("Total Tokens:", sum(tok_counter_train.values()))
print("<SEP>:", tok_counter_train["<SEP>"])

for tok, count in tok_counter_train.most_common(20):
    print(f"{tok:20s} {count}")

TRAIN
Total Tokens: 1643652
<SEP>: 440889
<SEP>                440889
<PAD>                316312
DUR:0.5              176880
DUR:0.2              146361
DUR:1                67849
REL_KOMA:31          56508
REL_KOMA:22          52492
REL_KOMA:0           40777
REL_KOMA:53          30446
REL_KOMA:13          24910
REL_KOMA:44          21869
<REST>               16795
REL_KOMA:36          16390
REL_KOMA:14          15545
REL_KOMA:8           15285
REL_KOMA:17          15216
REL_KOMA:5           15109
REL_KOMA:9           13792
DUR:0.8              12332
REL_KOMA:35          11704


In [31]:
# Hier wird das ocabulary gebaucht
# Makam = Conditioning-Token


SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<REST>", "<START>", "<END>", "<SEP>"]


def build_vocabulary(sequences):

    # Alle Tokens aus allen Sequenzen sammeln
    vocab_set = {
        tok
        for seq in sequences
        for tok in seq
    }

    vocab_set -= set(SPECIAL_TOKENS)
    vocab = SPECIAL_TOKENS + sorted(vocab_set)

    token_to_idx = {
        tok: idx
        for idx, tok in enumerate(vocab)
    }

    idx_to_token = {
        idx: tok
        for tok, idx in token_to_idx.items()
    }

    return vocab, token_to_idx, idx_to_token


# für Train


vocab, token_to_idx, idx_to_token = build_vocabulary(
    dataset_sequences_train
)

# Feste Spezialtoken-IDs
PAD_ID = token_to_idx["<PAD>"]
UNK_ID = token_to_idx["<UNK>"]
REST_ID = token_to_idx["<REST>"]
START_ID = token_to_idx["<START>"]
END_ID = token_to_idx["<END>"]
SEP_ID = token_to_idx["<SEP>"]

vocab_size = len(vocab)


with open("vocabulary.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=4)



print("Vocab Size:", vocab_size)
print("PAD_ID:", PAD_ID)
print("UNK_ID:", UNK_ID)
print(
    "Makam-Tokens im Vokabular:",
    sum(tok in MAKAM_TOKENS for tok in vocab)
)

Vocab Size: 136
PAD_ID: 0
UNK_ID: 1
Makam-Tokens im Vokabular: 0


In [32]:
# REL_koma Tokens werden gesammelt aus dem Vokabular
REL_KOMA_IDS = [
    idx for tok, idx in token_to_idx.items()
    if tok.startswith("REL_KOMA:")
]

print("REL_KOMA Tokens:", len(REL_KOMA_IDS))

REL_KOMA Tokens: 113


In [33]:
# Debug-Ausgabe

print("Vocabulary erstellt")
print("Vokabulargröße:", len(vocab))

print("\nBeispiel-Tokens:")
for tok in vocab[:15]:
    print(tok)



Vocabulary erstellt
Vokabulargröße: 136

Beispiel-Tokens:
<PAD>
<UNK>
<REST>
<START>
<END>
<SEP>
DUR:0.1
DUR:0.2
DUR:0.3
DUR:0.4
DUR:0.5
DUR:0.7
DUR:0.8
DUR:0.9
DUR:1


In [34]:
# Token-Sequenzen -> Integer-IDs
# Unbekannte Tokens werden zu <UNK>

def tokens_to_indices(sequences, token_to_idx, unk_token="<UNK>"):

    unk_id = token_to_idx[unk_token]

    return [
        [token_to_idx.get(tok, unk_id) for tok in seq]
        for seq in sequences
    ]


# TRAIN

indexed_sequences_train = tokens_to_indices(
    dataset_sequences_train,
    token_to_idx
)

print("TRAIN Sequenzen:", len(indexed_sequences_train))

if indexed_sequences_train:
    print(indexed_sequences_train[0][:50])
    print("Länge:", len(indexed_sequences_train[0]))


# VALIDATION

indexed_sequences_val = tokens_to_indices(
    dataset_sequences_val,
    token_to_idx
)

print("\nVALIDATION Sequenzen:", len(indexed_sequences_val))

if indexed_sequences_val:
    print(indexed_sequences_val[0][:50])
    print("Länge:", len(indexed_sequences_val[0]))


# TEST

indexed_sequences_test = tokens_to_indices(
    dataset_sequences_test,
    token_to_idx
)

print("\nTEST Sequenzen:", len(indexed_sequences_test))

if indexed_sequences_test:
    print(indexed_sequences_test[0][:50])
    print("Länge:", len(indexed_sequences_test[0]))


# UNK-Test

unk_count_val = sum(
    tok == UNK_ID
    for seq in indexed_sequences_val
    for tok in seq
)

unk_count_test = sum(
    tok == UNK_ID
    for seq in indexed_sequences_test
    for tok in seq
)

print("\nUNK Tokens Validation:", unk_count_val)
print("UNK Tokens Test:", unk_count_test)

TRAIN Sequenzen: 1602
[3, 73, 14, 5, 82, 14, 5, 105, 10, 5, 82, 10, 5, 73, 15, 5, 86, 14, 5, 82, 10, 5, 86, 10, 5, 92, 17, 5, 108, 10, 5, 101, 10, 5, 94, 10, 5, 92, 10, 5, 86, 10, 5, 82, 10, 5, 105, 10, 5, 73]
Länge: 1026

VALIDATION Sequenzen: 448
[3, 73, 14, 5, 94, 15, 5, 94, 15, 5, 92, 10, 5, 101, 7, 5, 94, 7, 5, 94, 7, 5, 92, 7, 5, 94, 10, 5, 101, 10, 5, 108, 10, 5, 101, 7, 5, 94, 7, 5, 92, 10, 5, 92, 10, 5, 92, 7, 5, 86]
Länge: 1026

TEST Sequenzen: 185
[3, 92, 7, 5, 95, 7, 5, 92, 7, 5, 86, 7, 5, 82, 10, 5, 86, 10, 5, 92, 14, 5, 95, 10, 5, 103, 7, 5, 108, 7, 5, 103, 7, 5, 95, 7, 5, 92, 14, 5, 86, 10, 5, 92, 7, 5, 95, 7, 5, 92]
Länge: 1026

UNK Tokens Validation: 51
UNK Tokens Test: 0


In [35]:
# NumPy Arrays wird vorbereitet

from torch.utils.data import Dataset, DataLoader, ConcatDataset

train_sequences_all = np.array(indexed_sequences_train, dtype=np.int32)
val_sequences_all   = np.array(indexed_sequences_val, dtype=np.int32)
test_sequences_all  = np.array(indexed_sequences_test, dtype=np.int32)

print("train:", train_sequences_all.shape)
print("val:", val_sequences_all.shape)
print("test:", test_sequences_all.shape)



# Reproduzierbarkeit

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
tf.random.set_seed(SEED)

print("A: arrays fertig")





# Augmentierungseinstellungen

augm_rate = 0.1  # Anteil der Noten, die verändert werden sollen
PAD_ID = token_to_idx["<PAD>"]
UNK_ID = token_to_idx["<UNK>"]



def augment_sequence(seq_ids):
    seq_len = len(seq_ids)

    tokens = [idx_to_token[int(i)] for i in seq_ids]
    new_tokens = []

    # letzte 5 vergangene Notenwerte und Dauern
    last_notes = []
    last_durs = []

    i = 0

    while i < len(tokens):
        tok = tokens[i]

        # Padding beendet echte Sequenz
        if tok == "<PAD>":
            break

        # START und END unverändert übernehmen
        if tok in ["<START>", "<END>"]:
            new_tokens.append(tok)
            i += 1
            continue

        # ein Event sammeln: REL_KOMA DUR <SEP>
        event = []

        while i < len(tokens):
            cur = tokens[i]

            if cur in ["<PAD>", "<END>"]:
                break

            event.append(cur)
            i += 1

            if cur == "<SEP>":
                break

        # ist das Event eine Note?
        is_note = len(event) > 0 and event[0].startswith("REL_KOMA:")

        if is_note:

            # Positionen holen
            pitch_idx = 0
            dur_idx = next(
                (j for j, t in enumerate(event) if t.startswith("DUR:")),
                None
            )

            # mit augm_rate entscheiden, ob diese Note verändert wird
            if random.random() < augm_rate:

                # zufällig eines von drei Verfahren wählen
                aug_type = random.choice(["rest", "duration", "pitch"])

                # 1) Note -> Rest
                if aug_type == "rest" and dur_idx is not None:
                    event = ["<REST>", event[dur_idx], "<SEP>"]

                # 2) Dauer aus den letzten 5 Noten übernehmen
                elif aug_type == "duration" and dur_idx is not None:
                    choices = [d for d in last_durs if d != event[dur_idx]]

                    if choices:
                        event[dur_idx] = random.choice(choices)

                # 3) Pitch aus den letzten 5 Noten übernehmen
                elif aug_type == "pitch":
                    choices = [p for p in last_notes if p != event[pitch_idx]]

                    if choices:
                        event[pitch_idx] = random.choice(choices)

            # aktuelle Note nach der Augmentation merken
            last_notes.append(event[pitch_idx])
            last_notes = last_notes[-5:]

            if dur_idx is not None:
                last_durs.append(event[dur_idx])
                last_durs = last_durs[-5:]

        # Event zur neuen Sequenz hinzufügen
        new_tokens.extend(event)

    # Tokens zurück zu IDs
    new_ids = [
        token_to_idx.get(tok, UNK_ID)
        for tok in new_tokens
    ]

    # auf Originallänge auffüllen
    if len(new_ids) < seq_len:
        new_ids.extend([PAD_ID] * (seq_len - len(new_ids)))

    return np.array(new_ids[:seq_len], dtype=np.int32)




# Input / Target Split


def split_input_target(seq):
    seq = np.asarray(seq, dtype=np.int32)
    return seq[:-1], seq[1:]


print("B: split fertig")

train: (1602, 1026)
val: (448, 1026)
test: (185, 1026)
A: arrays fertig
B: split fertig


In [36]:
# DataLoader

class SequenceDataset(Dataset):
    def __init__(self, sequences, makam_ids, augment=False):
        assert len(sequences) == len(makam_ids)
        self.sequences = sequences
        self.makam_ids = makam_ids
        self.augment = augment

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx].copy()

        if self.augment:
            seq = augment_sequence(seq)

        x, y = split_input_target(seq)

        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long),
            torch.tensor(self.makam_ids[idx], dtype=torch.long),
        )

In [37]:
class SequenceBatchGenerator(tf.keras.utils.Sequence):

    def __init__(self, sequences, makam_ids, batch_size, augment=False, shuffle=True):

        # Trainingsdaten speichern
        self.sequences = sequences
        self.makam_ids = np.array(makam_ids, dtype=np.int32)

        # Batchgröße
        self.batch_size = batch_size

        # Augmentation an/aus
        self.augment = augment

        # Reihenfolge mischen?
        self.shuffle = shuffle

        # Anzahl Original-Sequenzen
        self.n = len(sequences)

        # Wenn augment=True:
        # erste Hälfte = Original
        # zweite Hälfte = augmentierte Kopien
        if augment:
            self.indices = np.arange(self.n * 2)
        else:
            self.indices = np.arange(self.n)

        # am Anfang direkt mischen
        self.on_epoch_end()

    def __len__(self):

        # Wie viele Batches pro Epoche?
        return math.ceil(len(self.indices) / self.batch_size)

    def __getitem__(self, idx):

        # Welche Samples gehören zu diesem Batch?
        batch_ids = self.indices[
            idx * self.batch_size : (idx + 1) * self.batch_size
        ]

        x_tokens = []
        x_makam = []
        y_batch = []

        for virtual_i in batch_ids:

            # echten Index holen
            # z.B. 12003 -> 3
            real_i = virtual_i % self.n

            # Original-Sequenz kopieren
            seq = self.sequences[real_i].copy()

            # zweite Hälfte = augmentieren
            if self.augment and virtual_i >= self.n:
                seq = augment_sequence(seq)

            # Input / Target erzeugen
            x, y = split_input_target(seq)

            # Batch sammeln
            x_tokens.append(x)
            y_batch.append(y)
            x_makam.append(self.makam_ids[real_i])

        # Rückgabe für Keras
        return {
            "token_input": np.array(x_tokens, dtype=np.int32),
            "makam_input": np.array(x_makam, dtype=np.int32),
        }, np.array(y_batch, dtype=np.int32)

    def on_epoch_end(self):

        # nach jeder Epoche mischen
        if self.shuffle:
            np.random.shuffle(self.indices)

In [38]:
# TensorFlow Batch-Generatoren für Train / Val / Test

train_tf_dataset = SequenceBatchGenerator(
    sequences=train_sequences_all,
    makam_ids=dataset_makam_ids_train,
    batch_size=batch_size,
    augment=False,
    shuffle=True
)

val_tf_dataset = SequenceBatchGenerator(
    sequences=val_sequences_all,
    makam_ids=dataset_makam_ids_val,
    batch_size=batch_size,
    augment=False,
    shuffle=False
)

test_tf_dataset = SequenceBatchGenerator(
    sequences=test_sequences_all,
    makam_ids=dataset_makam_ids_test,
    batch_size=batch_size,
    augment=False,
    shuffle=False
)

In [39]:
xb_tf, yb_tf = train_tf_dataset[0]

print(xb_tf["token_input"].shape)
print(xb_tf["makam_input"].shape)
print(yb_tf.shape)

(16, 1025)
(16,)
(16, 1025)


In [40]:
# Vergleichsfunktion für Augmentierung vs Orginal

def compare_original_and_augmented(seq_ids, idx_to_token, max_events=20):

    def tokens_to_events(tokens):

        events = []
        current = []

        for tok in tokens:

            if tok == "<PAD>":
                break

            current.append(tok)

            if tok in ["<SEP>", "<END>"]:
                events.append(" ".join(current))
                current = []

        return events

    # ORIGINAL
    original_tokens = [
        idx_to_token[int(i)]
        for i in seq_ids
    ]

    # AUGMENTIERT
    augmented_ids = augment_sequence(seq_ids)

    augmented_tokens = [
        idx_to_token[int(i)]
        for i in augmented_ids
    ]

    original_events = tokens_to_events(original_tokens)
    augmented_events = tokens_to_events(augmented_tokens)

    print("=" * 80)
    print("ORIGINAL")
    print("=" * 80)

    for ev in original_events[:max_events]:
        print(ev)

    print("\n" + "=" * 80)
    print("AUGMENTIERT")
    print("=" * 80)

    for ev in augmented_events[:max_events]:
        print(ev)

In [41]:
compare_original_and_augmented(
    train_sequences_all[0],
    idx_to_token,
    max_events=150
)

ORIGINAL
<START> REL_KOMA:0 DUR:1 <SEP>
REL_KOMA:17 DUR:1 <SEP>
REL_KOMA:5 DUR:0.5 <SEP>
REL_KOMA:17 DUR:0.5 <SEP>
REL_KOMA:0 DUR:1.5 <SEP>
REL_KOMA:22 DUR:1 <SEP>
REL_KOMA:17 DUR:0.5 <SEP>
REL_KOMA:22 DUR:0.5 <SEP>
REL_KOMA:31 DUR:2 <SEP>
REL_KOMA:53 DUR:0.5 <SEP>
REL_KOMA:44 DUR:0.5 <SEP>
REL_KOMA:35 DUR:0.5 <SEP>
REL_KOMA:31 DUR:0.5 <SEP>
REL_KOMA:22 DUR:0.5 <SEP>
REL_KOMA:17 DUR:0.5 <SEP>
REL_KOMA:5 DUR:0.5 <SEP>
REL_KOMA:0 DUR:0.5 <SEP>
REL_KOMA:5 DUR:0.5 <SEP>
REL_KOMA:17 DUR:0.5 <SEP>
REL_KOMA:0 DUR:1.5 <SEP>
REL_KOMA:5 DUR:0.2 <SEP>
REL_KOMA:17 DUR:0.2 <SEP>
REL_KOMA:22 DUR:0.2 <SEP>
REL_KOMA:22 DUR:0.2 <SEP>
REL_KOMA:17 DUR:0.2 <SEP>
REL_KOMA:5 DUR:0.2 <SEP>
REL_KOMA:0 DUR:1 <SEP>
<REST> DUR:0.5 <SEP>
REL_KOMA:44 DUR:0.5 <SEP>
REL_KOMA:35 DUR:0.2 <SEP>
REL_KOMA:31 DUR:0.2 <SEP>
REL_KOMA:22 DUR:0.5 <SEP>
REL_KOMA:35 DUR:0.5 <SEP>
REL_KOMA:31 DUR:0.5 <SEP>
REL_KOMA:22 DUR:0.5 <SEP>
REL_KOMA:17 DUR:0.5 <SEP>
REL_KOMA:22 DUR:0.5 <SEP>
REL_KOMA:31 DUR:0.2 <SEP>
REL_KOMA:22 DUR:0.2 

In [42]:
# Debug-Ausgabe

# ersten Batch holen
xb, yb = train_tf_dataset[0]

print("token_input:", xb["token_input"].shape)
print("makam_input:", xb["makam_input"].shape)
print("y:", yb.shape)

print()

# erstes Sample anzeigen
print("ERSTES SAMPLE:")
print(" ".join(
    idx_to_token[int(i)]
    for i in xb["token_input"][0][:80]
))

print()

# letztes Sample anzeigen
print("LETZTES SAMPLE:")
print(" ".join(
    idx_to_token[int(i)]
    for i in xb["token_input"][-1][:80]
))

token_input: (16, 1025)
makam_input: (16,)
y: (16, 1025)

ERSTES SAMPLE:
<START> REL_KOMA:31 DUR:0.5 <SEP> REL_KOMA:22 DUR:0.5 <SEP> REL_KOMA:17 DUR:0.5 <SEP> REL_KOMA:22 DUR:0.5 <SEP> REL_KOMA:31 DUR:0.5 <SEP> REL_KOMA:35 DUR:0.5 <SEP> REL_KOMA:31 DUR:0.5 <SEP> REL_KOMA:22 DUR:0.5 <SEP> REL_KOMA:44 DUR:0.5 <SEP> REL_KOMA:35 DUR:0.5 <SEP> REL_KOMA:31 DUR:0.5 <SEP> REL_KOMA:22 DUR:0.5 <SEP> REL_KOMA:17 DUR:0.5 <SEP> REL_KOMA:5 DUR:0.5 <SEP> REL_KOMA:0 DUR:0.5 <SEP> REL_KOMA:5 DUR:0.5 <SEP> REL_KOMA:17 DUR:1.5 <SEP> REL_KOMA:22 DUR:0.5 <SEP> REL_KOMA:17 DUR:0.5 <SEP> REL_KOMA:5 DUR:0.5 <SEP> REL_KOMA:0 DUR:1 <SEP> <REST> DUR:1 <SEP> REL_KOMA:0 DUR:0.5 <SEP> REL_KOMA:5 DUR:0.5 <SEP> REL_KOMA:0 DUR:1 <SEP> REL_KOMA:-5 DUR:1 <SEP> REL_KOMA:0

LETZTES SAMPLE:
<START> REL_KOMA:48 DUR:4 <SEP> REL_KOMA:53 DUR:1 <SEP> REL_KOMA:62 DUR:1 <SEP> REL_KOMA:62 DUR:2 <SEP> REL_KOMA:62 DUR:1 <SEP> REL_KOMA:53 DUR:0.5 <SEP> REL_KOMA:44 DUR:0.5 <SEP> REL_KOMA:67 DUR:1 <SEP> REL_KOMA:62 DUR:0.5 <SEP> REL_KO

In [44]:
# Keras / TensorFlow Daten vorbereiten

X_tokens_train = train_sequences_all[:, :-1]
y_train = train_sequences_all[:, 1:]

X_tokens_val = val_sequences_all[:, :-1]
y_val = val_sequences_all[:, 1:]

X_tokens_test = test_sequences_all[:, :-1]
y_test = test_sequences_all[:, 1:]

X_makam_train = np.array(dataset_makam_ids_train, dtype=np.int32)
X_makam_val   = np.array(dataset_makam_ids_val, dtype=np.int32)
X_makam_test  = np.array(dataset_makam_ids_test, dtype=np.int32)

INPUT_LEN = X_tokens_train.shape[1]
num_makams = len(makam_to_id)

print("X_tokens_train:", X_tokens_train.shape)
print("y_train:", y_train.shape)

print("X_tokens_val:", X_tokens_val.shape)
print("y_val:", y_val.shape)

print("X_tokens_test:", X_tokens_test.shape)
print("y_test:", y_test.shape)

print("X_makam_train:", X_makam_train.shape)
print("X_makam_val:", X_makam_val.shape)
print("X_makam_test:", X_makam_test.shape)

print("INPUT_LEN:", INPUT_LEN)
print("num_makams:", num_makams)

X_tokens_train: (1602, 1025)
y_train: (1602, 1025)
X_tokens_val: (448, 1025)
y_val: (448, 1025)
X_tokens_test: (185, 1025)
y_test: (185, 1025)
X_makam_train: (1602,)
X_makam_val: (448,)
X_makam_test: (185,)
INPUT_LEN: 1025
num_makams: 56


In [53]:
# Layer: RMSNorm & GRU

class RMSNorm(tf.keras.layers.Layer):
    def __init__(self, d_model, eps=1e-8, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.eps = eps

    def build(self, input_shape):
        self.scale = self.add_weight(
            name="scale",
            shape=(self.d_model,),
            initializer="ones",
            trainable=True
        )

    def call(self, x):
        rms = tf.sqrt(tf.reduce_mean(tf.square(x), axis=-1, keepdims=True) + self.eps)
        return x / rms * self.scale

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model, "eps": self.eps})
        return config


class GRUFusion(tf.keras.layers.Layer):
    def __init__(self, d_model, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.gru = tf.keras.layers.GRU(d_model, return_sequences=True)

    def call(self, x_old, x_new, training=False):
        return self.gru(tf.concat([x_old, x_new], axis=-1), training=training)

    def get_config(self):
        config = super().get_config()
        config.update({"d_model": self.d_model})
        return config

In [54]:
# Rel. Position Bias + Self Attenation

class RelativePositionBias(tf.keras.layers.Layer):
    def __init__(self, n_heads, max_dist=max_dist, **kwargs):
        super().__init__(**kwargs)
        self.n_heads = n_heads
        self.max_dist = max_dist
        self.emb = tf.keras.layers.Embedding(2 * max_dist + 1, n_heads)

    def call(self, T):
        i = tf.range(T)[:, None]
        j = tf.range(T)[None, :]

        rel = tf.clip_by_value(j - i, -self.max_dist, self.max_dist)
        bias = self.emb(rel + self.max_dist)

        return tf.transpose(bias, [2, 0, 1])

    def get_config(self):
        config = super().get_config()
        config.update({"n_heads": self.n_heads, "max_dist": self.max_dist})
        return config


class RelPosSelfAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads, dropout=0.1, max_dist=max_dist, **kwargs):
        super().__init__(**kwargs)

        assert d_model % n_heads == 0, "d_model muss durch n_heads teilbar sein"

        self.d_model = d_model
        self.n_heads = n_heads
        self.dropout_rate = dropout
        self.max_dist = max_dist
        self.head_dim = d_model // n_heads

        self.qkv = tf.keras.layers.Dense(3 * d_model, use_bias=False)
        self.out = tf.keras.layers.Dense(d_model)
        self.dropout = tf.keras.layers.Dropout(dropout)
        self.rel_bias = RelativePositionBias(n_heads, max_dist=max_dist)

    def call(self, x, training=False):
        B = tf.shape(x)[0]
        T = tf.shape(x)[1]

        q, k, v = tf.split(self.qkv(x), 3, axis=-1)

        def split_heads(t):
            t = tf.reshape(t, [B, T, self.n_heads, self.head_dim])
            return tf.transpose(t, [0, 2, 1, 3])

        q, k, v = split_heads(q), split_heads(k), split_heads(v)

        scores = tf.matmul(q, k, transpose_b=True) * (self.head_dim ** -0.5)
        scores += self.rel_bias(T)[None, :, :, :]

        causal_mask = tf.linalg.band_part(tf.ones((T, T)), -1, 0)
        scores += (1.0 - causal_mask)[None, None, :, :] * -1e9

        attn = self.dropout(tf.nn.softmax(scores, axis=-1), training=training)

        out = tf.matmul(attn, v)
        out = tf.transpose(out, [0, 2, 1, 3])
        out = tf.reshape(out, [B, T, self.d_model])

        return self.out(out)

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "n_heads": self.n_heads,
            "dropout": self.dropout_rate,
            "max_dist": self.max_dist
        })
        return config

In [55]:
# Token-Gruppen und Makam-Anzahl Vorbereitung

REL_KOMA_IDS = [
    idx for tok, idx in token_to_idx.items()
    if tok.startswith("REL_KOMA:")
]

DUR_IDS = [
    idx for tok, idx in token_to_idx.items()
    if tok.startswith("DUR:")
]

REST_IDS = [REST_ID]

VALID_IDS_EXCEPT_PAD_UNK = [
    idx for idx in token_to_idx.values()
    if idx not in [PAD_ID, UNK_ID]
]

num_makams = len(makam_to_id)

print("REL_KOMA Tokens:", len(REL_KOMA_IDS))
print("DUR Tokens:", len(DUR_IDS))
print("REST Tokens:", len(REST_IDS))
print("Alle außer PAD/UNK:", len(VALID_IDS_EXCEPT_PAD_UNK))
print("Anzahl Makams:", num_makams)

REL_KOMA Tokens: 113
DUR Tokens: 17
REST Tokens: 1
Alle außer PAD/UNK: 134
Anzahl Makams: 56


In [56]:
# Musikalische Token-IDs ohne Special Tokens

MUSIC_IDS = [
    idx for idx in token_to_idx.values()
    if idx not in [PAD_ID, UNK_ID, SEP_ID, START_ID, END_ID]
]

print("MUSIC_IDS:", len(MUSIC_IDS))

MUSIC_IDS: 131


In [57]:
# Accuracy Metriken

class TokenGroupAccuracy(tf.keras.metrics.Metric):
    def __init__(self, token_ids, name="token_group_accuracy", **kwargs):
        super().__init__(name=str(name), **kwargs)

        self.token_ids_list = [int(x) for x in token_ids]
        self.token_ids = tf.constant(self.token_ids_list, dtype=tf.int64)

        self.correct = self.add_weight(
            name="correct_count",
            initializer="zeros"
        )
        self.total = self.add_weight(
            name="total_count",
            initializer="zeros"
        )

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.cast(y_true, tf.int64)
        y_pred_ids = tf.argmax(y_pred, axis=-1, output_type=tf.int64)

        mask = tf.reduce_any(
            tf.equal(y_true[..., None], self.token_ids),
            axis=-1
        )

        matches = tf.equal(y_true, y_pred_ids)
        matches = tf.logical_and(matches, mask)

        self.correct.assign_add(tf.reduce_sum(tf.cast(matches, tf.float32)))
        self.total.assign_add(tf.reduce_sum(tf.cast(mask, tf.float32)))

    def result(self):
        return tf.math.divide_no_nan(self.correct, self.total)

    def reset_state(self):
        self.correct.assign(0.0)
        self.total.assign(0.0)

    def get_config(self):
        return {
            "name": str(self.name),
            "token_ids": self.token_ids_list
        }

In [59]:
# Transformer Decoder Block
# Relative Attention + Feed Forward + GRU Fusion

class TransformerDecoderBlock(tf.keras.layers.Layer):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, max_distance=max_dist, **kwargs):
        super().__init__(**kwargs)

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_ff = d_ff
        self.dropout_rate = dropout
        self.max_distance = max_distance

        self.attn = RelPosSelfAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout,
            max_dist=max_distance
        )

        self.attn_dropout = tf.keras.layers.Dropout(dropout)
        self.gru1 = GRUFusion(d_model)
        self.norm1 = RMSNorm(d_model)

        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(d_ff, activation="relu"),
            tf.keras.layers.Dense(d_model),
        ])

        self.ffn_dropout = tf.keras.layers.Dropout(dropout)
        self.gru2 = GRUFusion(d_model)
        self.norm2 = RMSNorm(d_model)

    def call(self, x, training=False):
        attn_out = self.attn(x, training=training)
        attn_out = self.attn_dropout(attn_out, training=training)

        x = self.gru1(x, attn_out, training=training)
        x = self.norm1(x)

        ffn_out = self.ffn(x, training=training)
        ffn_out = self.ffn_dropout(ffn_out, training=training)

        x = self.gru2(x, ffn_out, training=training)
        x = self.norm2(x)

        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "d_model": self.d_model,
            "n_heads": self.n_heads,
            "d_ff": self.d_ff,
            "dropout": self.dropout_rate,
            "max_distance": self.max_distance,
        })
        return config

In [60]:
# TARREN Modell

d_model = 16
n_heads = 1
n_layers = 2
d_ff = 4 * d_model
dropout = 0.10
learning_rate = 0.001
weight_decay=0.00

def build_tarrean_keras(
    vocab_size, input_len, num_makams,
    d_model, n_heads, n_layers, d_ff,
    dropout, max_distance,
):
    token_input = tf.keras.Input((input_len,), dtype=tf.int32, name="token_input")
    makam_input = tf.keras.Input((), dtype=tf.int32, name="makam_input")

    token_emb = tf.keras.layers.Embedding(vocab_size, d_model, name="token_embedding")(token_input)
    makam_emb = tf.keras.layers.Embedding(num_makams, d_model, name="makam_embedding")(makam_input)

    makam_emb = tf.keras.layers.Reshape((1, d_model), name="makam_expand")(makam_emb)
    makam_emb = tf.keras.layers.Lambda(
        lambda m: tf.tile(m, [1, input_len, 1]),
        name="makam_repeat",
    )(makam_emb)

    x = tf.keras.layers.Add(name="token_plus_makam")([token_emb, makam_emb])
    x = tf.keras.layers.Dropout(dropout, name="embedding_dropout")(x)

    for i in range(n_layers):
        x = TransformerDecoderBlock(
            d_model, n_heads, d_ff,
            dropout=dropout,
            max_distance=max_distance,
            name=f"decoder_block_{i+1}",
        )(x)

    x = tf.keras.layers.Dense(d_model, name="final_projection")(x)
    x = tf.keras.layers.Dropout(dropout, name="output_dropout")(x)
    logits = tf.keras.layers.Dense(vocab_size, name="token_output")(x)

    return tf.keras.Model(
        inputs={"token_input": token_input, "makam_input": makam_input},
        outputs=logits,
        name="Tarrean_Transformer",
    )

In [61]:
# Model bauen

model = build_tarrean_keras(
    vocab_size=vocab_size,
    input_len=INPUT_LEN,
    num_makams=num_makams,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    d_ff=d_ff,
    dropout=dropout,
    max_distance=max_dist,
)

model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=learning_rate,
        weight_decay=weight_decay,
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True,
        ignore_class=PAD_ID,
    ),
    metrics=[
        TokenGroupAccuracy(REL_KOMA_IDS, name="acc_koma"),
        TokenGroupAccuracy(DUR_IDS, name="acc_dur"),
        TokenGroupAccuracy(REST_IDS, name="acc_rest"),
        TokenGroupAccuracy(VALID_IDS_EXCEPT_PAD_UNK, name="acc_no_pad_unk"),
        TokenGroupAccuracy(MUSIC_IDS, name="acc_music_only"),
    ],
)

model.summary()

Model: "Tarrean_Transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ makam_input (InputLayer)      │ (None)                    │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ makam_embedding (Embedding)   │ (None, 16)                │             896 │ makam_input[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_input (InputLayer)      │ (None, 1025)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ makam_expand (Reshape)        │ (None, 1, 16)             │               0 │ makam_embedding[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding (Embedding)   │ (None, 1025, 16)          │           2,176 │ token_input[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ makam_repeat (Lambda)         │ (None, 1025, 16)          │               0 │ makam_expand[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_plus_makam (Add)        │ (None, 1025, 16)          │               0 │ token_embedding[0][0],     │
│                               │                           │                 │ makam_repeat[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_dropout (Dropout)   │ (None, 1025, 16)          │               0 │ token_plus_makam[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_block_1               │ (None, 1025, 16)          │           8,513 │ embedding_dropout[0][0]    │
│ (TransformerDecoderBlock)     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_block_2               │ (None, 1025, 16)          │           8,513 │ decoder_block_1[0][0]      │
│ (TransformerDecoderBlock)     │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ final_projection (Dense)      │ (None, 1025, 16)          │             272 │ decoder_block_2[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ output_dropout (Dropout)      │ (None, 1025, 16)          │               0 │ final_projection[0][0]     │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_output (Dense)          │ (None, 1025, 136)         │           2,312 │ output_dropout[0][0]       │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 22,682 (88.60 KB)

 Trainable params: 22,682 (88.60 KB)

 Non-trainable params: 0 (0.00 B)

In [90]:
# Trainings-Callbacks: Checkpoint, EarlyStopping, LR-Scheduler

runs_dir = "runs"
os.makedirs(runs_dir, exist_ok=True)

def make_callbacks(run_name, patience=3):
    run_dir = os.path.join(runs_dir, run_name)
    os.makedirs(run_dir, exist_ok=True)

    return [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(
                run_dir,
                "best_model.weights.h5"
            ),
            monitor="val_loss",
            mode="min",
            save_best_only=True,
            save_weights_only=True,
        ),

        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            mode="min",
            patience=patience,
            restore_best_weights=True,
        ),

        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        ),
    ]

In [91]:
# Experimentstart

def run_experiment(run_name, cfg):
    print(f"\nStart Experiment: {run_name}")
    print(cfg)

    model = build_tarrean_keras(
        vocab_size=vocab_size,
        input_len=INPUT_LEN,
        num_makams=num_makams,
        d_model=cfg["d_model"],
        n_heads=cfg["n_heads"],
        n_layers=cfg["n_layers"],
        d_ff=cfg["d_ff"],
        dropout=cfg["dropout"],
        max_distance=cfg["max_distance"],
    )

    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=cfg["lr"],
            weight_decay=cfg["weight_decay"],
        ),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            TokenGroupAccuracy(REL_KOMA_IDS, "acc_koma"),
            TokenGroupAccuracy(DUR_IDS, "acc_dur"),
            TokenGroupAccuracy(REST_IDS, "acc_rest"),
            TokenGroupAccuracy(VALID_IDS_EXCEPT_PAD_UNK, "acc_no_pad_unk"),
            TokenGroupAccuracy(MUSIC_IDS, "acc_music_only"),
        ],
    )

    history = model.fit(
        {"token_input": X_tokens_train, "makam_input": X_makam_train},
        y_train,
        validation_data=(
            {"token_input": X_tokens_val, "makam_input": X_makam_val},
            y_val,
        ),
        epochs=cfg["epochs"],
        batch_size=cfg["batch_size"],
        callbacks=make_callbacks(run_name, cfg.get("patience", 3)),
        verbose=2,
    )

    return {
        "run": run_name,
        "best_val_loss": float(np.min(history.history["val_loss"])),
        "params": model.count_params(),
        "history": history.history,
        "model": model,
    }

In [92]:
# Experiment-Konfigurationen

EPOCHS= 50
PATIENCE = 8

MANUAL_SETUPS = [
    dict(
        tag="Experiment: REL-1023",
        d_model=128,
        n_heads=2,
        n_layers=2,
        d_ff=128 * 4,
        lr=0.001,
        dropout=0.10,
        weight_decay=1e-6,
    )
]

experiment_configs = {
    s["tag"]: {
        **s,
        "batch_size": batch_size,
        "epochs": EPOCHS,
        "patience": PATIENCE,
        "max_distance": max_dist,
    }
    for s in MANUAL_SETUPS
}

print("Configs erzeugt:", len(experiment_configs))

Configs erzeugt: 1


In [93]:
# Experimente ausführen 

results = []

for run_name, cfg in experiment_configs.items():
    result = run_experiment(run_name, cfg)

    test_metrics = result["model"].evaluate(
        {"token_input": X_tokens_test, "makam_input": X_makam_test},
        y_test,
        batch_size=cfg["batch_size"],
        verbose=2,
        return_dict=True,
    )

    result["test_loss"] = float(test_metrics["loss"])
    del result["model"]

    results.append(result)

results_df = (
    pd.DataFrame(results)
    .sort_values("best_val_loss")
    .reset_index(drop=True)
)

results_df


Start Experiment: max_dist = 64
{'tag': 'max_dist = 64', 'd_model': 16, 'n_heads': 1, 'n_layers': 1, 'd_ff': 512, 'lr': 0.001, 'dropout': 0.1, 'weight_decay': 1e-06, 'batch_size': 16, 'epochs': 3, 'patience': 8, 'max_distance': 256}
Epoch 1/3
101/101 - 177s - 2s/step - acc_dur: 0.2937 - acc_koma: 0.0046 - acc_music_only: 0.1490 - acc_no_pad_unk: 0.4030 - acc_rest: 0.0033 - loss: 2.6987 - val_acc_dur: 0.5844 - val_acc_koma: 9.8606e-05 - val_acc_music_only: 0.2919 - val_acc_no_pad_unk: 0.5270 - val_acc_rest: 0.0000e+00 - val_loss: 1.7016 - learning_rate: 0.0010
Epoch 2/3
101/101 - 157s - 2s/step - acc_dur: 0.5689 - acc_koma: 0.0853 - acc_music_only: 0.3252 - acc_no_pad_unk: 0.5492 - acc_rest: 6.5496e-04 - loss: 1.4689 - val_acc_dur: 0.6357 - val_acc_koma: 0.1257 - val_acc_music_only: 0.3781 - val_acc_no_pad_unk: 0.5845 - val_acc_rest: 0.0000e+00 - val_loss: 1.2719 - learning_rate: 0.0010
Epoch 3/3
101/101 - 162s - 2s/step - acc_dur: 0.6006 - acc_koma: 0.1983 - acc_music_only: 0.3955 - a

,run,best_val_loss,params,history,test_loss
0,max_dist = 64,1.123526,28953,"{'acc_dur': [0.2937394380569458, 0.56891751289...",1.063548


In [94]:
# Loss-Plot speichern

plots_dir = os.path.join(runs_dir, "plots")
os.makedirs(plots_dir, exist_ok=True)

save_path = os.path.join(plots_dir, "Loss_Plot.png")

history = results[-1]["history"]

fig, ax = plt.subplots(figsize=(6, 4))

ax.plot(history["loss"], label="train_loss")
ax.plot(history["val_loss"], label="val_loss")

ax.set(title="Loss Plot", xlabel="Epochen", ylabel="Loss")
ax.legend()
ax.grid(True)

fig.savefig(save_path, format="png", dpi=150, bbox_inches="tight")
plt.close(fig)

print("Gespeichert:", save_path)
print("Dateigröße:", os.path.getsize(save_path), "Bytes")

C:\Users\oezde\AppData\Local\Temp\ipykernel_30192\334223999.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [95]:
# Bestes Modell laden 


best_run = results_df.iloc[0]["run"]
best_cfg = experiment_configs[best_run]

best_model = build_tarrean_keras(
    vocab_size=vocab_size,
    input_len=INPUT_LEN,
    num_makams=num_makams,
    d_model=best_cfg["d_model"],
    n_heads=best_cfg["n_heads"],
    n_layers=best_cfg["n_layers"],
    d_ff=best_cfg["d_ff"],
    dropout=best_cfg["dropout"],
    max_distance=best_cfg["max_distance"],
)

best_weights_path = os.path.join(
    runs_dir,
    best_run,
    "best_model.weights.h5"
)

best_model.load_weights(best_weights_path)

In [96]:
# Bestes Modell laden und auf Testdaten auswerten

best_run = results_df.iloc[0]["run"]
best_cfg = experiment_configs[best_run]

best_model = build_tarrean_keras(
    vocab_size=vocab_size,
    input_len=INPUT_LEN,
    num_makams=num_makams,
    d_model=best_cfg["d_model"],
    n_heads=best_cfg["n_heads"],
    n_layers=best_cfg["n_layers"],
    d_ff=best_cfg["d_ff"],
    dropout=best_cfg["dropout"],
    max_distance=best_cfg["max_distance"],
)

best_model.load_weights(
    os.path.join(runs_dir, best_run, "best_model.weights.h5")
)

best_model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=best_cfg["lr"],
        weight_decay=best_cfg["weight_decay"],
    ),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[
        TokenGroupAccuracy(REL_KOMA_IDS, "acc_koma"),
        TokenGroupAccuracy(DUR_IDS, "acc_dur"),
        TokenGroupAccuracy(REST_IDS, "acc_rest"),
        TokenGroupAccuracy(VALID_IDS_EXCEPT_PAD_UNK, "acc_no_pad_unk"),
        TokenGroupAccuracy(MUSIC_IDS, "acc_music_only"),
    ],
)

test_metrics = best_model.evaluate(
    {"token_input": X_tokens_test, "makam_input": X_makam_test},
    y_test,
    batch_size=best_cfg["batch_size"],
    verbose=2,
    return_dict=True,
)

print("Bester Run:", best_run)
print("Test:", test_metrics)

28/28 - 29s - 1s/step - acc_dur: 0.6138 - acc_koma: 0.2901 - acc_music_only: 0.4468 - acc_no_pad_unk: 0.6302 - acc_rest: 0.0000e+00 - loss: 1.0635
Bester Run: max_dist = 64
Test: {'acc_dur': 0.6138085126876831, 'acc_koma': 0.29006755352020264, 'acc_music_only': 0.4468126893043518, 'acc_no_pad_unk': 0.6302390694618225, 'acc_rest': 0.0, 'loss': 1.0635480880737305}


In [97]:
# Summary und History für besten Run speichern

best_row = results_df.iloc[0]
best_run = best_row["run"]
best_cfg = experiment_configs[best_run]
best_result = next(r for r in results if r["run"] == best_run)
history = best_result["history"]

test_metrics = best_model.evaluate(
    {"token_input": X_tokens_test, "makam_input": X_makam_test},
    y_test,
    batch_size=best_cfg["batch_size"],
    verbose=2,
    return_dict=True,
)

best_info = {
    "best_run": best_run,
    "objective": "val_loss",
    "pitch_representation": "relative",
    "hyperparameters": best_cfg,
    "vocab_size": vocab_size,
    "seq_len": seq_len,
    "input_len": INPUT_LEN,
    "num_makams": num_makams,
    "total_params": int(best_result["params"]),
    "best_history_values": {
        "best_val_loss": float(min(history["val_loss"])),
    },
    "last_history_values": {
        k: float(v[-1]) for k, v in history.items() if v
    },
    "test_values": {
        k: float(v) for k, v in test_metrics.items()
    },
}

with open(os.path.join(plots_dir, "best_run_summary.json"), "w", encoding="utf-8") as f:
    json.dump(best_info, f, indent=4)

with open(os.path.join(plots_dir, "history.json"), "w", encoding="utf-8") as f:
    json.dump(history, f, indent=4)

print("Gespeichert: best_run_summary.json, history.json")
print("Bester Run:", best_run)
print("Test:", test_metrics)

28/28 - 26s - 938ms/step - acc_dur: 0.6138 - acc_koma: 0.2901 - acc_music_only: 0.4468 - acc_no_pad_unk: 0.6302 - acc_rest: 0.0000e+00 - loss: 1.0635
Gespeichert: best_run_summary.json, history.json
Bester Run: max_dist = 64
Test: {'acc_dur': 0.6138085126876831, 'acc_koma': 0.29006755352020264, 'acc_music_only': 0.4468126893043518, 'acc_no_pad_unk': 0.6302390694618225, 'acc_rest': 0.0, 'loss': 1.0635480880737305}
